# FreshRetailNet-50K — Data Ingestion & Preprocessing for MMF

## Dataset Overview

**FreshRetailNet-50K** is an open benchmark dataset for perishable grocery demand forecasting,
released by [Dingdong Inc.](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)
It contains **50,000 store-product time series** of daily sales data from **898 stores** across
**18 major cities**, covering **865 perishable SKUs** over a 90-day period (Mar–Jun 2024).

| Property | Value |
|---|---|
| **Source** | [HuggingFace — Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K) |
| **License** | **Creative Commons Attribution 4.0 International (CC BY 4.0)** |
| **Rows** | ~4.85M (4.5M train + 350K eval) |
| **Time Series** | 50,000 store-product combinations |
| **Date Range** | 90 consecutive days (Mar 28 – Jun 25, 2024) |
| **Stores** | 898 across 18 cities |
| **Products** | 865 perishable SKUs |
| **External Regressors** | Discount, holiday, promotions, precipitation, temperature, humidity, wind |

### License & Attribution

This dataset is licensed under **CC BY 4.0**, which permits sharing, adaptation, and commercial use — including publication in blog posts — provided that appropriate credit is given.

> **Citation:** Dingdong Inc. (2025). FreshRetailNet-50K: A Stockout-Annotated Censored Demand Dataset for Latent Demand Recovery and Forecasting in Fresh Retail. https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K

### Why This Dataset for MMF?

- **Dense time series:** Every store-product combo has 90 consecutive daily data points — no sparsity
- **Supply chain relevance:** Perishable grocery demand forecasting is a classic supply chain problem
- **Built-in external regressors:** Discount, holiday, promotions, and weather features ready for MMF
- **Hierarchical structure:** City → Store → Product category hierarchy for grouped forecasting
- **Scale:** 50,000 series (sampled to 1,000 for this demo)

## 1. Setup — Install Dependencies & Create Schema

In [0]:
%pip install datasets
dbutils.library.restartPython()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 13.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 12.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.3/193.3 kB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 47.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 74.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 81.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
# CREATE CATALOG IF NOT EXISTS mmf
# Workaround: UC validates metastore storage root before the existence check,
# so we guard with an explicit check to avoid the error when the catalog already exists.
existing_catalogs = {r.catalog for r in spark.sql("SHOW CATALOGS").collect()}
if "mmf" not in existing_catalogs:
    spark.sql("CREATE CATALOG mmf")
    print("Created catalog: mmf")
else:
    print("Catalog mmf already exists — skipping creation")

Catalog mmf already exists — skipping creation


In [0]:
# CREATE SCHEMA IF NOT EXISTS mmf.fresh_retail_net
# (Catalog mmf is expected to already exist — created manually or by cell 4)
spark.sql("CREATE SCHEMA IF NOT EXISTS mmf.fresh_retail_net")
print("Schema mmf.fresh_retail_net: ready")

Schema mmf.fresh_retail_net: ready


## 2. Download the Dataset from HuggingFace

In [0]:
import tempfile, os

# HuggingFace datasets needs a writable cache dir (serverless blocks /root/.cache)
hf_cache = tempfile.mkdtemp()
os.environ["HF_HOME"] = hf_cache
os.environ["HF_DATASETS_CACHE"] = hf_cache

from datasets import load_dataset

hf_ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K", cache_dir=hf_cache)
print(hf_ds)

/databricks/python_shell/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /tmp/tmpmgqsjdot, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)


README.md:   0%|          | 0.00/4.87k [00:00<?, ?B/s]

/databricks/python_shell/dbruntime/huggingface_patches/datasets.py:14: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


data/train.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

data/eval.parquet:   0%|          | 0.00/8.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 4500000
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 350000
    })
})


## 3. Convert to Spark and Save Raw Data

The dataset has `hours_sale` (list of 24 floats) and `hours_stock_status` (list of 24 ints) columns.
We drop these array columns for the Delta table since MMF operates at daily granularity.

In [0]:
import pandas as pd

# Combine train and eval splits
pdf_train = hf_ds["train"].to_pandas()
pdf_eval = hf_ds["eval"].to_pandas()
pdf_train["split"] = "train"
pdf_eval["split"] = "eval"
pdf_raw = pd.concat([pdf_train, pdf_eval], ignore_index=True)

print(f"Train rows:  {len(pdf_train):,}")
print(f"Eval rows:   {len(pdf_eval):,}")
print(f"Total rows:  {len(pdf_raw):,}")
print(f"Columns:     {list(pdf_raw.columns)}")
print(f"Date range:  {pdf_raw['dt'].min()} — {pdf_raw['dt'].max()}")
print(f"Unique stores:   {pdf_raw['store_id'].nunique():,}")
print(f"Unique products: {pdf_raw['product_id'].nunique():,}")
print(f"Unique combos:   {pdf_raw.groupby(['store_id','product_id']).ngroups:,}")

Train rows:  4,500,000
Eval rows:   350,000
Total rows:  4,850,000
Columns:     ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'split']
Date range:  2024-03-28 — 2024-07-02
Unique stores:   898
Unique products: 865
Unique combos:   50,000


In [0]:
# Drop hourly array columns (not needed for daily MMF) and convert date
pdf_raw = pdf_raw.drop(columns=["hours_sale", "hours_stock_status"])
pdf_raw["dt"] = pd.to_datetime(pdf_raw["dt"]).dt.date

df_raw = spark.createDataFrame(pdf_raw)
df_raw.printSchema()

root
 |-- city_id: long (nullable = true)
 |-- store_id: long (nullable = true)
 |-- management_group_id: long (nullable = true)
 |-- first_category_id: long (nullable = true)
 |-- second_category_id: long (nullable = true)
 |-- third_category_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- dt: date (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- stock_hour6_22_cnt: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- holiday_flag: integer (nullable = true)
 |-- activity_flag: integer (nullable = true)
 |-- precpt: double (nullable = true)
 |-- avg_temperature: double (nullable = true)
 |-- avg_humidity: double (nullable = true)
 |-- avg_wind_level: double (nullable = true)
 |-- split: string (nullable = true)



In [0]:
df_raw.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.daily_sales_raw")
print("Saved raw data to mmf.fresh_retail_net.daily_sales_raw")

Saved raw data to mmf.fresh_retail_net.daily_sales_raw


## 4. Explore the Raw Data

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT store_id) AS unique_stores,
  COUNT(DISTINCT product_id) AS unique_products,
  COUNT(DISTINCT city_id) AS unique_cities,
  MIN(dt) AS min_date,
  MAX(dt) AS max_date,
  DATEDIFF(MAX(dt), MIN(dt)) + 1 AS total_days,
  ROUND(AVG(sale_amount), 4) AS avg_daily_sales,
  ROUND(AVG(discount), 4) AS avg_discount
FROM mmf.fresh_retail_net.daily_sales_raw

total_rows,unique_stores,unique_products,unique_cities,min_date,max_date,total_days,avg_daily_sales,avg_discount
4850000,898,865,18,2024-03-28,2024-07-02,97,1.0126,0.9119


In [0]:
%sql
-- Check density: how many days does each store-product combo have?
SELECT
  MIN(day_count) AS min_days,
  ROUND(AVG(day_count), 1) AS avg_days,
  MAX(day_count) AS max_days
FROM (
  SELECT store_id, product_id, COUNT(*) AS day_count
  FROM mmf.fresh_retail_net.daily_sales_raw
  GROUP BY store_id, product_id
)

min_days,avg_days,max_days
97,97.0,97


## 5. Sample to 1,000 Store-Product Combos & Prepare for MMF

We sample 1,000 random store-product combinations from the training split,
then build the MMF-ready table using only data from those combos (both train and eval splits).

In [0]:
from pyspark.sql import functions as F

df = spark.table("mmf.fresh_retail_net.daily_sales_raw")

# Build unique_id on full dataset first
df = df.withColumn("unique_id", F.concat_ws("_", F.col("store_id"), F.col("product_id")))

# Get combos that exist in BOTH train and eval splits (required for MMF train/score alignment)
train_ids = df.filter(F.col("split") == "train").select("unique_id").distinct()
eval_ids = df.filter(F.col("split") == "eval").select("unique_id").distinct()
shared_ids = train_ids.intersect(eval_ids)
print(f"Combos in both splits: {shared_ids.count():,}")

# Deterministic sample: hash-based ordering to ensure reproducibility
sampled_ids = (
    shared_ids
    .orderBy(F.hash("unique_id"))
    .limit(1000)
)
sampled_ids.count()  # Materialize the cache
print(f"Sampled combos: {sampled_ids.count()}")

Combos in both splits: 50,000
Sampled combos: 1000


In [0]:
# Filter to sampled combos (both train and eval data)
df_sampled = (
    df.join(sampled_ids, on="unique_id", how="inner")
    .withColumnRenamed("dt", "ds")
    .withColumnRenamed("sale_amount", "y")
)

# Verify both splits are present for all sampled series
train_count = df_sampled.filter(F.col("split") == "train").select("unique_id").distinct().count()
eval_count = df_sampled.filter(F.col("split") == "eval").select("unique_id").distinct().count()
print(f"Sampled rows: {df_sampled.count():,}")
print(f"Train series: {train_count}  |  Eval series: {eval_count}")
assert train_count == eval_count == 1000, f"Mismatch! train={train_count}, eval={eval_count}"

Sampled rows: 97,000
Train series: 1000  |  Eval series: 1000


## 6. Split into Train & Score Tables for MMF

MMF expects two separate tables:
- **Train table** — historical data **with** the target column (`y`)
- **Score table** — future dates **without** the target column, but with external regressors

The FreshRetailNet dataset already provides a natural split:
- `split == "train"` → 90 days of history (used for training)
- `split == "eval"` → 7 days of future data (used for scoring)

In [0]:
regressor_cols = [
    "discount",
    "holiday_flag",
    "activity_flag",
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level",
]

grouping_cols = [
    "city_id",
    "store_id",
    "product_id",
    "management_group_id",
    "first_category_id",
    "second_category_id",
    "third_category_id",
]

# Train table: historical data WITH the target (y)
df_train = (
    df_sampled
    .filter(F.col("split") == "train")
    .select("unique_id", "ds", "y", *regressor_cols, *grouping_cols, "stock_hour6_22_cnt")
)

# Score table: future dates WITHOUT the target, but with regressors
df_score = (
    df_sampled
    .filter(F.col("split") == "eval")
    .select("unique_id", "ds", *regressor_cols, *grouping_cols)
)

print(f"Train rows: {df_train.count():,}  |  Series: {df_train.select('unique_id').distinct().count()}")
print(f"Score rows: {df_score.count():,}  |  Series: {df_score.select('unique_id').distinct().count()}")

# Verify date ranges don't overlap
train_max = df_train.agg(F.max("ds")).collect()[0][0]
score_min = df_score.agg(F.min("ds")).collect()[0][0]
print(f"\nTrain date range: {df_train.agg(F.min('ds')).collect()[0][0]} — {train_max}")
print(f"Score date range: {score_min} — {df_score.agg(F.max('ds')).collect()[0][0]}")
print(f"Gap check: train ends {train_max}, score starts {score_min} (should be consecutive)")

Train rows: 90,000  |  Series: 1000
Score rows: 7,000  |  Series: 1000

Train date range: 2024-03-28 — 2024-06-25
Score date range: 2024-06-26 — 2024-07-02
Gap check: train ends 2024-06-25, score starts 2024-06-26 (should be consecutive)


In [0]:
df_train.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.demand_train")
df_score.write.mode("overwrite").saveAsTable("mmf.fresh_retail_net.demand_score")
print("Saved train data to mmf.fresh_retail_net.demand_train")
print("Saved score data to mmf.fresh_retail_net.demand_score")

Saved train data to mmf.fresh_retail_net.demand_train
Saved score data to mmf.fresh_retail_net.demand_score


## 7. Validate the Output

In [0]:
%sql
SELECT 'train' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT unique_id) AS num_series,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date,
  DATEDIFF(MAX(ds), MIN(ds)) + 1 AS total_days,
  ROUND(AVG(y), 4) AS avg_daily_demand,
  ROUND(SUM(CASE WHEN y > 0 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS pct_nonzero
FROM mmf.fresh_retail_net.demand_train
UNION ALL
SELECT 'score' AS table_name,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT unique_id) AS num_series,
  MIN(ds) AS min_date,
  MAX(ds) AS max_date,
  DATEDIFF(MAX(ds), MIN(ds)) + 1 AS total_days,
  NULL AS avg_daily_demand,
  NULL AS pct_nonzero
FROM mmf.fresh_retail_net.demand_score

table_name,total_rows,num_series,min_date,max_date,total_days,avg_daily_demand,pct_nonzero
train,90000,1000,2024-03-28,2024-06-25,90,0.9763,95.4
score,7000,1000,2024-06-26,2024-07-02,7,null,null


In [0]:
%sql
-- Top 20 series by total demand (train data)
SELECT
  unique_id,
  COUNT(*) AS total_days,
  ROUND(SUM(y), 2) AS total_demand,
  ROUND(AVG(y), 4) AS avg_daily_demand,
  SUM(CASE WHEN y > 0 THEN 1 ELSE 0 END) AS active_days
FROM mmf.fresh_retail_net.demand_train
GROUP BY unique_id
ORDER BY total_demand DESC
LIMIT 20

unique_id,total_days,total_demand,avg_daily_demand,active_days
496_267,90,1689.3,18.77,87
630_267,90,1512.2,16.8022,90
290_267,90,1219.0,13.5444,87
831_267,90,1127.6,12.5289,87
822_117,90,657.6,7.3067,87
637_589,90,638.1,7.09,90
154_691,90,615.4,6.8378,89
187_117,90,563.4,6.26,90
557_70,90,511.0,5.6778,90
312_691,90,481.5,5.35,90


In [0]:
%sql
-- Sample time series for the top-demand series (train only)
SELECT ds, y, discount, holiday_flag, activity_flag, avg_temperature
FROM mmf.fresh_retail_net.demand_train
WHERE unique_id = (
  SELECT unique_id
  FROM mmf.fresh_retail_net.demand_train
  GROUP BY unique_id
  ORDER BY SUM(y) DESC
  LIMIT 1
)
ORDER BY ds

ds,y,discount,holiday_flag,activity_flag,avg_temperature
2024-03-28,3.8,0.0,0,0,16.77
2024-03-29,2.4,0.0,0,0,16.52
2024-03-30,6.0,0.0,1,0,16.86
2024-03-31,3.3,0.0,1,0,16.93
2024-04-01,3.0,0.0,0,0,16.51
2024-04-02,3.0,0.0,0,0,17.64
2024-04-03,3.0,0.0,0,0,17.89
2024-04-04,4.3,0.0,1,0,16.64
2024-04-05,3.6,0.0,1,0,16.66
2024-04-06,5.3,0.0,1,0,17.79


## Output Tables

### `mmf.fresh_retail_net.demand_train` — Training data (1,000 series x 90 days)

| Column | Type | MMF Role | Description |
|---|---|---|---|
| `unique_id` | STRING | **Group ID** | `{store_id}_{product_id}` — the forecasting unit |
| `ds` | DATE | **Time column** | Date (daily, no gaps) |
| `y` | DOUBLE | **Target** | Daily sales amount (normalized) |
| `discount` | DOUBLE | Regressor | Discount rate (1.0 = no discount) |
| `holiday_flag` | INT | Regressor | Holiday indicator |
| `activity_flag` | INT | Regressor | Promotion indicator |
| `precpt` | DOUBLE | Regressor | Precipitation |
| `avg_temperature` | DOUBLE | Regressor | Average temperature |
| `avg_humidity` | DOUBLE | Regressor | Average humidity |
| `avg_wind_level` | DOUBLE | Regressor | Average wind level |
| `stock_hour6_22_cnt` | INT | Info | Out-of-stock hours (6am–10pm) |
| `city_id` | INT | Grouping | City |
| `store_id` | INT | Grouping | Store |
| `product_id` | INT | Grouping | Product |
| `management_group_id` | INT | Grouping | Management group |
| `first_category_id` | INT | Grouping | Product category L1 |
| `second_category_id` | INT | Grouping | Product category L2 |
| `third_category_id` | INT | Grouping | Product category L3 |

### `mmf.fresh_retail_net.demand_score` — Scoring data (1,000 series x 7 days)

Same columns as train **except `y` is excluded** (this is what MMF will predict).

### Example MMF `run_forecast` call:
```python
run_forecast(
    spark=spark,
    train_data="mmf.fresh_retail_net.demand_train",
    scoring_data="mmf.fresh_retail_net.demand_score",
    scoring_output="mmf.fresh_retail_net.scoring_output",
    evaluation_output="mmf.fresh_retail_net.evaluation_output",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="D",
    prediction_length=7,
    backtest_length=14,
    stride=7,
    metric="smape",
    train_predict_ratio=2,
    dynamic_future_numerical=["discount", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"],
    dynamic_future_categorical=["holiday_flag", "activity_flag"],
)
```